[Énoncés : Série 6 — Sommes préfixes, recherche et tri](../serie6_prefixes_recherche_tri.ipynb)

# Corrigé — Série 6 — Sommes préfixes, recherche et tri

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 30 — Compter les nombres pairs entre deux indices

**Idée** : les **sommes préfixes**. On calcule une seule fois `prefixe[i]` = nombre de pairs dans les `i` premiers éléments. Ensuite, chaque requête se résout par une soustraction : `prefixe[d + 1] - prefixe[g]`. On paie O(n) une fois au lieu de O(n) par requête.

**Complexité** : O(n + q). Compter dans chaque intervalle coûterait O(n × q).

In [ ]:
def requetes_pairs(nombres: list[int], requetes: list[list[int]]) -> list[int]:
    # prefixe[i] = nombre d'éléments pairs parmi nombres[0], ..., nombres[i - 1]
    prefixe = [0]
    for x in nombres:
        prefixe.append(prefixe[-1] + (1 if x % 2 == 0 else 0))
    # pairs entre g et d inclus = (pairs avant d + 1) − (pairs avant g)
    return [prefixe[d + 1] - prefixe[g] for g, d in requetes]


verifier(requetes_pairs, [
    (([2, 3, 4, 6, 7], [[0, 2], [1, 4], [3, 3]]), [2, 2, 1]),
    (([1, 3, 5], [[0, 2]]), [0]),
    (([2], [[0, 0]]), [1]),
    (([1, 2, 3, 4], []), []),   # aucune requête
    (([0, -2, 5, 8], [[0, 3], [2, 2], [1, 1]]), [3, 0, 1]),   # 0 et -2 sont pairs
])

In [ ]:
# Test de vitesse
chrono(requetes_pairs, list(range(10_000)), [[0, 9_999]] * 10_000, attendu=[5_000] * 10_000)

## Exercice 31 — Racine carrée entière

**Idée** : la **recherche dichotomique** sur la réponse. Comme `x * x` grandit avec `x`, on coupe l'intervalle des candidats en deux à chaque étape : si `milieu²` est ≤ n, la réponse est à droite (milieu compris), sinon à gauche. Piège classique : `milieu = (bas + haut) // 2` avec `bas = milieu` boucle à l'infini quand `haut = bas + 1`.

**Complexité** : O(log n), soit une quarantaine d'étapes pour 10¹².

In [ ]:
def racine_entiere(n: int) -> int:
    bas, haut = 0, n                      # la réponse est entre 0 et n
    while bas < haut:
        milieu = (bas + haut + 1) // 2    # arrondi vers le haut, sinon boucle infinie quand haut = bas + 1
        if milieu * milieu <= n:
            bas = milieu                  # milieu convient : la réponse est milieu ou plus grande
        else:
            haut = milieu - 1             # milieu est trop grand
    return bas


verifier(racine_entiere, [
    ((17,), 4),
    ((144,), 12),
    ((0,), 0),
    ((1,), 1),
    ((2,), 1),
    ((15,), 3),
    ((999999,), 999),
])

In [ ]:
# Test de vitesse
chrono(racine_entiere, 10**12, attendu=10**6)

## Exercice 32 — Compter les valeurs dans plusieurs intervalles

**Idée** : la liste est triée, donc les éléments de `[minimum, maximum]` forment un bloc contigu. Deux recherches dichotomiques donnent les bords du bloc. En Python, le module `bisect` le fait déjà : `bisect_right(valeurs, maximum) - bisect_left(valeurs, minimum)`.

**Complexité** : O(q × log n).

In [ ]:
def compter_dans_plages(valeurs: list[int], plages: list[list[int]]) -> list[int]:
    def nb_inferieurs(x):
        # nombre d'éléments strictement inférieurs à x, par recherche dichotomique
        bas, haut = 0, len(valeurs)
        while bas < haut:
            milieu = (bas + haut) // 2
            if valeurs[milieu] < x:
                bas = milieu + 1
            else:
                haut = milieu
        return bas
    # éléments dans [minimum, maximum] = (éléments < maximum + 1) − (éléments < minimum)
    return [nb_inferieurs(maximum + 1) - nb_inferieurs(minimum) for minimum, maximum in plages]


verifier(compter_dans_plages, [
    (([-3, -1, -1, 2, 5, 5], [[-1, 5], [0, 4], [6, 10]]), [5, 1, 0]),
    (([], [[0, 10]]), [0]),   # liste vide
    (([1, 2, 3], [[1, 3], [2, 2], [-5, 0], [4, 9]]), [3, 1, 0, 0]),
    (([7, 7, 7], [[7, 7]]), [3]),
    (([1, 3, 5, 7], [[2, 6]]), [2]),
])

In [ ]:
# Test de vitesse
chrono(compter_dans_plages, list(range(10_000)), [[2_000, 7_999]] * 10_000, attendu=[6_000] * 10_000)

## Exercice 33 — Fusionner les intervalles

**Idée** : après un **tri par début**, deux intervalles fusionnables sont forcément voisins dans la liste. On parcourt une fois en comparant chaque intervalle au dernier du résultat : s'il commence avant `fin + 1`, on étend ; sinon on en commence un nouveau. Piège : prendre `max(...)` pour la nouvelle fin (un intervalle peut être inclus dans le précédent).

**Complexité** : O(n log n), à cause du tri.

In [ ]:
def fusionner_intervalles(intervalles: list[list[int]]) -> list[list[int]]:
    resultat = []
    for debut, fin in sorted(intervalles):                   # trier par début
        if resultat and debut <= resultat[-1][1] + 1:        # chevauche ou touche le dernier
            resultat[-1][1] = max(resultat[-1][1], fin)      # on l'étend
        else:
            resultat.append([debut, fin])                    # sinon, nouvel intervalle
    return resultat


verifier(fusionner_intervalles, [
    (([[5, 7], [1, 2], [3, 4], [9, 11], [10, 10]],), [[1, 7], [9, 11]]),
    (([],), []),
    (([[1, 1]],), [[1, 1]]),
    (([[1, 3], [5, 6]],), [[1, 3], [5, 6]]),   # il manque le 4 : pas de fusion
    (([[1, 10], [2, 3], [4, 5]],), [[1, 10]]),   # intervalles inclus dans un autre
    (([[6, 8], [1, 4], [5, 5]],), [[1, 8]]),
    (([[-5, -3], [-2, 0], [2, 2]],), [[-5, 0], [2, 2]]),
])

## Exercice 34 — Les k valeurs les plus fréquentes

**Idée** : un dict compteur, puis un tri des valeurs distinctes avec la clé `(-fréquence, valeur)`, et on garde les `k` premières. C'est l'exercice 6 avec des nombres.

**Complexité** : O(n + d log d), avec d le nombre de valeurs distinctes.

In [ ]:
def top_k_frequents(nombres: list[int], k: int) -> list[int]:
    compte = {}
    for x in nombres:
        compte[x] = compte.get(x, 0) + 1
    classement = sorted(compte, key=lambda x: (-compte[x], x))   # fréquence ↓, puis valeur ↑
    return classement[:k]


verifier(top_k_frequents, [
    (([5, 2, 5, 2, 3, 2, 4], 2), [2, 5]),
    (([1], 1), [1]),
    (([4, 4, 1, 1, 3], 2), [1, 4]),   # égalité : la plus petite valeur d'abord
    (([4, 4, 1, 1, 3], 3), [1, 4, 3]),
    (([7, 8, 9], 0), []),
    (([-1, -1, 2, 2, 2], 1), [2]),
])

## Exercice 35 — Plus longue portion de somme nulle

**Idée** : sommes cumulées + dict. Si la somme cumulée vaut la même chose aux indices `i` et `j`, alors les éléments entre les deux s'annulent. Pour chaque somme cumulée, on retient le **premier** indice où elle apparaît ; la plus longue portion nulle qui se termine en `i` va de cet indice + 1 jusqu'à `i`. L'entrée `{0: -1}` gère les portions qui commencent au début.

**Complexité** : O(n). Tester toutes les portions coûterait O(n²).

In [ ]:
def longueur_max_somme_nulle(nombres: list[int]) -> int:
    premiere_fois = {0: -1}   # somme cumulée -> premier indice où on l'a obtenue (0 « avant le début »)
    somme = 0
    meilleur = 0
    for i, x in enumerate(nombres):
        somme += x
        if somme in premiere_fois:
            # même somme cumulée qu'à l'indice premiere_fois[somme] : entre les deux, la somme est 0
            meilleur = max(meilleur, i - premiere_fois[somme])
        else:
            premiere_fois[somme] = i      # on ne garde que la PREMIÈRE fois (pour la plus longue portion)
    return meilleur


verifier(longueur_max_somme_nulle, [
    (([1, -1, 3, -3, 4],), 4),
    (([],), 0),
    (([1, 2, 3],), 0),   # aucune portion de somme nulle
    (([0],), 1),
    (([5, -5],), 2),
    (([2, -2, 2, -2, 1],), 4),
    (([1, 2, -3, 3, -3],), 5),   # toute la liste
    (([3, 4, -7, 1, 2, -6, 4],), 4),   # la portion [4, -7, 1, 2]
])

In [ ]:
# Test de vitesse
chrono(longueur_max_somme_nulle, [(i % 7) - 3 for i in range(10_000)], attendu=9_997)

---

[Énoncés de la série](../serie6_prefixes_recherche_tri.ipynb)